# Steerling-8B Instruct: Chunk-to-Concept Attribution

Concept attribution explains **why** the model generated each token, in terms of concepts. Steerling captures this **during generation itself**: each
token's attribution comes from the exact forward pass that committed it, using the
partially-masked context the model actually saw — which is what makes it faithful by design.

For each predicted token $y_t$, Steerling's concept heads decompose the output logit
into per-concept contributions:

$$C(c_i, y_t) = \sigma(\text{logit}_i) \cdot (e_i \cdot W_{y_t})$$

Contributions sum to the token's logit, up to a residual $\varepsilon_t$ not explained
by any concept (we verify this numerically in Section 4).

Consecutive tokens are grouped into **chunks**. To score a chunk, we normalize each
position's contributions (so every token votes equally) then average:

$$\text{chunk\_score}(i) = \frac{1}{T} \sum_{t \in \text{chunk}} \frac{C(c_i, y_t)}{\sum_j |C(c_j, y_t)| + |\varepsilon_t|}$$

**Requirements:** Interpretable Steerling model, GPU with >= 18 GB VRAM

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig
from steerling.attribution.concept_attribution import (
    FaithfulConceptAttributor, find_chunk_boundaries, chunk_attribution,
)

PURPLE = "#675BF2"
PURPLE_LIGHT = "#ceb4fe"
COLOR_MAP = {"known": PURPLE, "discovered": PURPLE_LIGHT}

## 1. Load Model & Concept Labels

In [ ]:
model_id = "guidelabs/steerling-8b-instruct"

model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

print(generator)
print(f"Interpretable: {generator.is_interpretable}")

## 2. Create Attributor

In [ ]:
attributor = FaithfulConceptAttributor(generator)

## 3. Generate & Attribute

Apply the chat template to format the prompt, then pass to the attributor.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": "What is a diffusion language model?"},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

config = GenerationConfig(
    max_new_tokens=256,
    steps=256,
    seed=42,
    stop_tokens=[tokenizer.convert_tokens_to_ids("<|eot_id|>")],
)

gen_output, chunk_results = attributor.attribute(prompt, config)

print(f"Generated ({gen_output.generated_tokens} tokens):")
print(gen_output.text)

## 4. Explore the Attribution

The attributor stores per-token concept contributions in `last_attribution`. Let's look at the
raw data before plotting.

In [ ]:
attr = attributor.last_attribution

print(f"Attributed positions:           {attr.known_contributions.shape[1]}")
print(f"Top-k known concepts per token: {attr.known_contributions.shape[-1]}")
print(f"Top-k discovered per token:     {attr.unk_contributions.shape[-1]}")

# Verify decomposition: contributions should sum to target logits
verification = attr.verify()
print(f"\nDecomposition check: max error = {verification.max_abs_error:.6f} (pass={verification.passed})")

In [ ]:
# Top concepts for a single token position
pos = gen_output.prompt_tokens  # first generated token
token_str = generator.tokenizer.decode([gen_output.tokens[pos].item()])
print(f"Token at position {pos}: {token_str!r}\n")

# Show top-5 known concept contributions at this position
contribs = attr.known_contributions[0, pos]  # (k_known,)
indices = attr.known_indices[0, pos]
top_vals, top_idx = contribs.abs().topk(min(5, contribs.shape[0]))

for i in top_idx:
    cid = int(indices[i])
    label = attributor.labels.label(cid, "known")
    print(f"  {contribs[i]:+.4f}  {label}")

## 5. Chunk-Level Visualization

Each concept's contribution is its **fraction of total logit mass** averaged across the chunk.
Because contributions are normalized per token and then averaged, individual values are small
(typically < 2%) — what matters is the **relative ranking**, not the absolute numbers.

**Epsilon** is the percentage of logit mass not explained by any concept. Lower epsilon means
the concepts account for more of the model's prediction.

Known concepts in purple, discovered in light purple.

In [ ]:
def plot_contributions(entries, title="Concept Contributions", eps_pct=0.0, top_k=20):
    """Horizontal bar chart of top concept contributions."""
    display = sorted(entries, key=lambda e: e["contribution"], reverse=True)[:top_k]

    labels = [e["label"] for e in display]
    values = [e["contribution"] for e in display]
    colors = [COLOR_MAP[e["type"]] for e in display]

    fig, ax = plt.subplots(figsize=(10, max(3, len(labels) * 0.35)))
    y = np.arange(len(labels))
    ax.barh(y, values, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Fraction of total |logit| mass")
    ax.set_title(f"{title}  |  epsilon = {eps_pct:.1f}%", fontsize=11)
    ax.legend(
        handles=[Patch(fc=COLOR_MAP[t], label=t.title()) for t in ["known", "discovered"]],
        loc="lower right", fontsize=8,
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
tokens = gen_output.tokens
prompt_len = gen_output.prompt_tokens
eoc_id = getattr(generator.tokenizer, "endofchunk_token_id", None)
eot_id = getattr(generator.tokenizer, "eot_id", None)
stop = [eot_id] if eot_id is not None else None

chunks = find_chunk_boundaries(
    tokens.tolist(),
    eoc_id=eoc_id if eoc_id is not None else -1,
    start_index=prompt_len,
    stop_ids=stop,
    include_final_chunk=True,
)
attr = attributor.last_attribution

for s, e in chunks:
    entries, eps_pct = chunk_attribution(
        attr, s, e, batch=0,
        concept_labels=attributor.labels,
        num_known_concepts=attributor._num_known_concepts,
    )
    text_preview = generator.tokenizer.decode(tokens[s:e].tolist(), skip_special_tokens=True)[:100]

    print(f"\n=== Chunk [{s}, {e}) | epsilon = {eps_pct:.1f}% ===")
    print(f"  Text: {text_preview!r}")
    top = sorted(entries, key=lambda e: e["contribution"], reverse=True)[:10]
    for entry in top:
        print(f"  {entry['type']:<12s} {entry['contribution']:6.2%}  {entry['label']}")

    plot_contributions(
        entries,
        title=f'"{text_preview[:60]}..."',
        eps_pct=eps_pct,
        top_k=20,
    )